In [1]:
import numpy as np
import torch
from PIL import Image
from transformers import AutoProcessor, Gemma3nForConditionalGeneration

# Load Gemma 3n model and processor
model_id = "google/gemma-3n-e4b-it"

print("Loading Gemma 3n model...")
model = Gemma3nForConditionalGeneration.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.bfloat16
).eval()

processor = AutoProcessor.from_pretrained(model_id)
print("Model loaded successfully!")


2025-07-01 22:35:21.613087: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751434521.630053  120417 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751434521.635110  120417 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1751434521.648528  120417 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1751434521.648552  120417 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1751434521.648554  120417 computation_placer.cc:177] computation placer alr

Loading Gemma 3n model...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

RuntimeError: Unknown model (mobilenetv5_300m_enc)

In [3]:
from transformers import CLIPTokenizer, CLIPTextModel
import torch
import torch.nn.functional as F

# Load CLIP text model for similarity comparison
clip_model_name = "openai/clip-vit-base-patch32"
clip_tokenizer = CLIPTokenizer.from_pretrained(clip_model_name)
clip_text_model = CLIPTextModel.from_pretrained(clip_model_name).cuda()

# Load and process image using Gemma 3n
image_path = "/home/jasonx/Pictures/Screenshots/towel_C.png"
image = Image.open(image_path).convert('RGB')

# Create messages in Gemma 3n format
question_text = 'Your role is to determine the status of the grey towel in this image. For each of the eval axes, give a concise answer. 1. Number of folds. 2. Number of corners hidden. 3. Is the towel laid flat on the table? 4. Tell me why. Go.'

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful vision assistant that analyzes images with precision."}]
    },
    {
        "role": "user", 
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question_text}
        ]
    }
]

# Process inputs using Gemma 3n processor
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True, 
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

# Generate response
print("Generating response...")
with torch.inference_mode():
    generation = model.generate(
        **inputs, 
        max_new_tokens=300, 
        do_sample=True, 
        temperature=0.7,
        top_p=0.9
    )
    generation = generation[0][input_len:]

# Decode response
response = processor.decode(generation, skip_special_tokens=True)

print(f"User: {question_text}")
print(f"Assistant: {response}")

# Extract the part before "3." for analysis
response_parsed = response.split("3.")[0] if "3." in response else response
print(f"========== Response parsed: =========== {response_parsed}")

# Define reference texts for comparison
reference_texts = [
    "yes, laid flat",
    "no, not flat"
]

# Encode all texts using CLIP
def encode_text(text):
    inputs = clip_tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cuda")
    with torch.no_grad():
        text_features = clip_text_model(**inputs).pooler_output
    return F.normalize(text_features, dim=-1)

# Get embeddings for similarity comparison
response_embedding = encode_text(response_parsed)
reference_embeddings = [encode_text(ref) for ref in reference_texts]

# Compute cosine similarities
similarities = []
for i, ref_embedding in enumerate(reference_embeddings):
    similarity = torch.cosine_similarity(response_embedding, ref_embedding, dim=-1).item()
    similarities.append(similarity)
    print(f"Similarity to '{reference_texts[i]}': {similarity:.4f}")

# Determine final verdict based on similarity scores
if similarities[0] > similarities[1]:
    verdict = "YES"
    confidence = similarities[0]
else:
    verdict = "NO" 
    confidence = similarities[1]

print(f"\nFinal Verdict: {verdict} (confidence: {confidence:.4f})")

User: Your role is to determine the status of the grey towel in this image: <image>. For each of the eval axes, give a concise answer. 1. Number of folds. 2. Number of corners hidden. 3. Is the towel laid flat on the table? 4. Tell me why. Go.
Assistant: 1. Number of folds: The grey towel in the image appears to have one visible fold, which starts from the bottom left corner and extends to the top right corner of the towel. The rest of the towel appears to be in a more unfolded state, displaying two visible corners.

2. Number of corners hidden: From the visible portion of the towel, two corners are clearly visible, while two others are hidden underneath the folded part. The additional two corners would be on the ends of the towel, one on the top right corner and one on the bottom left corner, but these are not visible in the image.

3. Is the towel laid flat on the table: Yes, the towel is laid flat on the table. This can be determined by observing the consistent texture and pattern o

AttributeError: module 'dspy' has no attribute 'HFModel'